# EmpowerLens - DistilBERT multi-label, wellally.tech tutorial recipe

Runs the recipe from
[wellally.tech/blog/python-cognitive-distortion-transformer-tutorial](https://www.wellally.tech/blog/python-cognitive-distortion-transformer-tutorial)
on the **real** annotated data instead of the tutorial's 15-row toy CSV.

**The recipe, kept verbatim:** `distilbert-base-uncased`,
`problem_type="multi_label_classification"` (plain BCE, no class weighting),
`lr=2e-5`, `batch_size=8`, `epochs=10`, `weight_decay=0.01`, eval every epoch,
`load_best_model_at_end`, sigmoid with a fixed **0.5** threshold, scored with
micro-F1 / ROC-AUC / accuracy.

**What we changed, and why:**

| Tutorial | Here | Why |
|---|---|---|
| 15 hand-written rows | `Annotated_data.csv` via `data/splits/` (2,024 train / 253 val / 253 test) | 15 rows cannot produce a number that means anything |
| `dataset.train_test_split(test_size=0.2)` each run | the committed frozen splits | project rule: splits are immutable, so runs stay comparable |
| `evaluation_strategy`, `tokenizer=`, `return_all_scores` | `eval_strategy`, `processing_class=`, `top_k=None` | transformers 5.x renamed all three |
| ROC-AUC on 0/1 predictions | ROC-AUC on probabilities (tutorial's version kept alongside) | AUC over binarized labels throws away the ranking it exists to measure |
| "accuracy" as a headline | reported as `subset_accuracy`, F1 is the headline | on a 10-column label matrix, `accuracy_score` = all 10 columns right at once |

The 10 label columns are the union of `Dominant Distortion` and
`Secondary Distortion (Optional)`; an all-zero row means *No Distortion*.
`test.csv` is only opened in the last section, by `src/evaluate.py`.

All logic lives in `src/tutorial_distilbert.py` - this notebook only drives it
and displays the results.

## 0. Environment

Skip the next cell when running locally from a clone. On **Kaggle**: Settings ->
Accelerator **GPU**, Internet **On**, then run it.

In [ ]:
# --- Kaggle only: clone the repo and install the transformer stack ---
# REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
# BRANCH   = "nayab-space"
# !rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
# %cd empowerlens
# !pip install -q -r requirements-transformer.txt

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# Work from the repo root no matter where the notebook was launched from.
here = Path.cwd()
if (here / "src" / "tutorial_distilbert.py").exists():
    ROOT = here
elif (here.parent / "src" / "tutorial_distilbert.py").exists():
    ROOT = here.parent          # launched from notebooks/
else:
    raise SystemExit(f"Cannot find src/tutorial_distilbert.py from {here}")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows OpenMP guard
import torch, transformers, pandas as pd, numpy as np

# Prefer the project venv's interpreter when this notebook runs on a kernel
# that isn't it; fall back to whatever is running us (Kaggle, Colab, venv).
PY = sys.executable
venv_py = ROOT / "venv" / "Scripts" / "python.exe"
if venv_py.exists() and "venv" not in PY.lower():
    PY = str(venv_py)

CUDA = torch.cuda.is_available()
print(f"python      : {PY}")
print(f"torch       : {torch.__version__}  |  transformers: {transformers.__version__}")
print(f"device      : {'cuda - ' + torch.cuda.get_device_name(0) if CUDA else 'cpu'}")

## 1. The data we are actually training on

Straight from the frozen splits - no reshuffling, no regeneration.

In [ ]:
from src.data import DISTORTIONS
from src.tutorial_distilbert import load_split, get_labels, ML_COLS

splits = {name: load_split("data/splits", name) for name in ("train", "val", "test")}
Y = {name: get_labels(df) for name, df in splits.items()}

manifest = json.loads(Path("data/splits/split_manifest.json").read_text())
print("rows per split:", {k: len(v) for k, v in splits.items()})
print("split random_state:", manifest.get("random_state", "?"), "\n")

dist = pd.DataFrame(
    {name: Y[name].sum(axis=0).astype(int) for name in ("train", "val", "test")},
    index=DISTORTIONS,
)
dist.loc["(no distortion: all-zero row)"] = [
    int((Y[n].sum(axis=1) == 0).sum()) for n in ("train", "val", "test")
]
dist["train_%"] = (100 * dist["train"] / len(splits["train"])).round(1)
display(dist)

lab_per_row = Y["train"].sum(axis=1)
print(f"labels per training row: mean {lab_per_row.mean():.2f}  |  "
      f"0 labels: {(lab_per_row==0).sum()}  1: {(lab_per_row==1).sum()}  "
      f"2: {(lab_per_row==2).sum()}")
print("\nRarest classes drive macro-F1 - watch these in the per-class table later:")
print(dist["train"][DISTORTIONS].sort_values().head(3).to_string())

## 2. Configure the run

The defaults below **are** the tutorial's hyperparameters. The only thing this
cell decides for you is how much you can afford to run: 10 epochs x 512 tokens
is roughly 15 min/seed on a T4 but many hours on CPU, so the CPU path shortens
it and says so.

`SEEDS = [42, 1337, 2024]` is the project's three-seed protocol (results are
reported as mean +/- std). Start with one seed to see it work.

In [ ]:
MODEL      = "distilbert-base-uncased"
TASK       = "multilabel"      # binary | multiclass | multilabel
SEEDS      = [42]              # -> [42, 1337, 2024] for the full protocol
LR         = 2e-5              # tutorial
BATCH_SIZE = 8                 # tutorial
WEIGHT_DEC = 0.01              # tutorial
THRESHOLD  = 0.5               # tutorial
OUT        = "results_tutorial_distilbert"

if CUDA:
    EPOCHS, MAX_LEN = 10, 512          # tutorial values, feasible on a GPU
    est = "~15 min per seed on a T4"
else:
    EPOCHS, MAX_LEN = 3, 256           # CPU: shortened so it finishes today
    est = "~60-90 min per seed on CPU - use a GPU for the full 10 x 512 run"

print(f"model={MODEL} seeds={SEEDS} epochs={EPOCHS} lr={LR} bs={BATCH_SIZE} "
      f"max_len={MAX_LEN} threshold={THRESHOLD}")
print("estimated:", est)
if not CUDA:
    print("\nNOTE: EPOCHS/MAX_LEN differ from the tutorial (10/512) because this "
          "is CPU.\n      Numbers below are therefore a lower bound, not the "
          "recipe's ceiling.")

### Optional: a 2-minute smoke test first

64 rows, 1 epoch. Proves the plumbing runs; the metrics it prints are noise.

In [ ]:
smoke = subprocess.run(
    [PY, "-m", "src.tutorial_distilbert", "--smoke", "--max-length", "128",
     "--out", "results_tutorial_distilbert_smoke", "--no-demo"],
    cwd=ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(smoke.stdout[-1500:] if smoke.returncode == 0
      else smoke.stdout[-3000:] + smoke.stderr[-3000:])
print("\nsmoke test:", "PASSED" if smoke.returncode == 0 else "FAILED")

## 3. Train

Streams the training log live. Each seed trains, picks its best epoch by **val
micro-F1**, then writes metrics, per-class CSVs and a checkpoint.

In [ ]:
cmd = [PY, "-m", "src.tutorial_distilbert",
       "--model", MODEL,
       "--seeds", ",".join(str(s) for s in SEEDS),
       "--epochs", str(EPOCHS),
       "--lr", str(LR),
       "--batch-size", str(BATCH_SIZE),
       "--weight-decay", str(WEIGHT_DEC),
       "--threshold", str(THRESHOLD),
       "--max-length", str(MAX_LEN),
       "--task", TASK,
       "--out", OUT]
print(" ".join(cmd), "\n")

proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True,
                        encoding="utf-8", errors="replace", bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)

## 4. Results - validation set

Headline is **micro-F1** (what the tutorial optimises). `macro_f1` weights all
ten distortions equally and is the harder, more honest number on this label
distribution.

In [ ]:
out = Path(OUT)
summary  = pd.read_csv(out / f"val_summary_{TASK}.csv", index_col=0)
per_seed = pd.read_csv(out / f"val_metrics_per_seed_{TASK}.csv", index_col=0)

print(f"Validation results - {len(per_seed)} seed(s): {list(per_seed.index)}\n")
display(summary[["mean_pm_std"]] if len(per_seed) > 1 else per_seed.T.round(4))

m = summary["mean"]
print(f'''
headline    micro-F1        {m["micro_f1"]:.3f}
            macro-F1        {m["macro_f1"]:.3f}   (all 10 classes weighted equally)
            weighted-F1     {m["weighted_f1"]:.3f}
ranking     ROC-AUC micro   {m["roc_auc_micro"]:.3f}   (on probabilities)
            ROC-AUC micro   {m["roc_auc_micro_tutorial"]:.3f}   (tutorial's version, on 0/1 preds)
strictness  subset accuracy {m["subset_accuracy"]:.3f}   (all 10 columns correct at once)
            hamming loss    {m["hamming_loss"]:.3f}
behaviour   labels fired    {m["mean_labels_predicted"]:.2f} per row vs {m["mean_labels_true"]:.2f} actual
''')
if m["mean_labels_predicted"] < 0.5 * m["mean_labels_true"]:
    print("The model is under-firing: unweighted BCE on rare positives collapses "
          "toward predicting nothing.\nThat is the recipe's main weakness - see section 6b.")

# Same probabilities, per-class thresholds swept on val instead of a flat 0.5.
tuned = pd.read_csv(out / f"val_summary_tuned_{TASK}.csv", index_col=0)["mean"]
cmp = pd.DataFrame({"@ fixed 0.5": m, "@ tuned": tuned}).loc[
    ["micro_f1", "macro_f1", "roc_auc_micro", "mean_labels_predicted"]]
cmp["delta"] = cmp["@ tuned"] - cmp["@ fixed 0.5"]
print("\nThreshold alone, no retraining (ROC-AUC cannot move - it never used "
      "the threshold):")
display(cmp.round(3))

### Per-class breakdown

`support` is the number of val rows carrying that label - a class with 4
positives out of 253 can post F1 = 0.00 from a single miss.

In [ ]:
pc = pd.read_csv(out / f"per_class_val_mean_{TASK}.csv", index_col=0)
display(pc.round(3).sort_values("f1", ascending=False))

dead = pc.index[pc["f1"] == 0].tolist()
if dead:
    print(f"{len(dead)}/10 classes never predicted correctly at threshold "
          f"{THRESHOLD}: {', '.join(dead)}")

png = out / f"per_class_val_f1_{TASK}.png"
try:
    from IPython.display import Image
    if png.exists():
        display(Image(str(png)))
except ImportError:
    print("chart written to", png)

### Learning curve

Ten epochs on 2k rows overfits well before the end - this is where you see it,
and which epoch `load_best_model_at_end` actually restored.

In [ ]:
import matplotlib.pyplot as plt

hist_files = sorted(out.glob("epoch_history_*.csv"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for f in hist_files:
    h = pd.read_csv(f)
    seed = f.stem.split("_")[-1]
    axes[0].plot(h["epoch"], h["eval_loss"], marker="o", label=f"seed {seed}")
    axes[1].plot(h["epoch"], h["eval_micro_f1"], marker="o", label=f"micro-F1 {seed}")
    axes[1].plot(h["epoch"], h["eval_macro_f1"], marker="s", ls="--", label=f"macro-F1 {seed}")
    best = h.loc[h["eval_micro_f1"].idxmax()]
    print(f"seed {seed}: best epoch {best['epoch']:.0f} "
          f"(val micro-F1 {best['eval_micro_f1']:.3f}) of {len(h)}")
axes[0].set(xlabel="epoch", ylabel="val loss", title="Validation loss")
axes[1].set(xlabel="epoch", ylabel="F1", title="Validation F1", ylim=(0, 1))
for a in axes:
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Inference - the tutorial's `predict.py`

The three sentences from the tutorial, plus room for your own.

In [ ]:
from transformers import pipeline
from src.tutorial_distilbert import DEMO_TEXTS

ckpt = Path("checkpoints") / f"tutorial_{TASK}_{MODEL.split('/')[-1]}_{SEEDS[-1]}"
# top_k=None is the transformers 5.x spelling of return_all_scores=True
clf = pipeline("text-classification", model=str(ckpt), tokenizer=str(ckpt),
               top_k=None, device=0 if CUDA else -1)

MY_TEXTS = [
    "If I don't get this right the whole semester is ruined.",
]

for text in DEMO_TEXTS + MY_TEXTS:
    scores = sorted(clf(text, truncation=True)[0], key=lambda d: d["score"], reverse=True)
    fired = [s for s in scores if s["score"] > THRESHOLD]
    print(f"\n{text!r}")
    if fired:
        for s in fired:
            print(f"   {s['label']:<22} {s['score']:.4f}")
    else:
        print(f"   (nothing above {THRESHOLD}) top 3: " +
              ", ".join(f"{s['label']} {s['score']:.3f}" for s in scores[:3]))

## 6. Test set

The one place `test.csv` is read, and only through `src/evaluate.py` - the
module the project designates for it. Run this **once**, after you have stopped
changing the recipe; every tuning decision above was made on val.

`--max-labels 0` means "no cap on how many labels may fire", which is the
tutorial's behaviour (it just thresholds at 0.5).

In [ ]:
for seed in SEEDS:
    ck = Path("checkpoints") / f"tutorial_{TASK}_{MODEL.split('/')[-1]}_{seed}"
    r = subprocess.run([PY, "-m", "src.evaluate", "--checkpoint", str(ck),
                        "--max-labels", "0", "--out", OUT],
                       cwd=ROOT, capture_output=True, text=True,
                       encoding="utf-8", errors="replace")
    print(r.stdout.strip()[-600:] or r.stderr[-1500:])

rows = []
for f in sorted(Path(OUT).glob("eval_*multilabel*.json")):
    d = json.loads(f.read_text())          # nested: splits -> val/test -> metrics
    for split_name, block in d["splits"].items():
        rows.append({"seed": d["meta"]["seed"], "split": split_name,
                     **{k: v for k, v in block["metrics"].items()
                        if k in ("macro_f1", "micro_f1", "weighted_f1")}})
test_tbl = pd.DataFrame(rows)
display(test_tbl.round(3))

if not test_tbl.empty and {"val", "test"} <= set(test_tbl["split"]):
    v = test_tbl[test_tbl.split == "val"]["micro_f1"].mean()
    t = test_tbl[test_tbl.split == "test"]["micro_f1"].mean()
    print(f"\nval micro-F1 {v:.3f} -> test micro-F1 {t:.3f}  "
          f"(drop of {v - t:+.3f}; some gap is expected, val is what we selected on)")

## 6b. Loss ablation - isolating the two fixes

The tutorial's weakness is that ~95% of the gradient on a rare class says "no",
so probabilities never reach 0.5. There are two independent remedies, and this
section measures them separately:

- **Change the loss** - alters what the model *learns*.
- **Change the threshold** - alters only where the decision line sits on the
  *same* probabilities. No retraining; it is a post-hoc rescoring.

Four losses, all sigmoid-based, all scoring the 10 labels independently:

| `--loss` | mechanism |
|---|---|
| `bce` | unweighted `BCEWithLogitsLoss` - the tutorial's |
| `pos_bce` | + `pos_weight = negatives/positives` per class (19.0x for `all_or_nothing`) |
| `focal` | `FocalLoss(gamma=2)` on top of `pos_weight` - also damps *easy* examples |
| `asl` | `AsymmetricLoss` - separate gammas for positives/negatives, replaces `pos_weight` |

Each is trained once and scored at both threshold settings, so this is 4
trainings for 8 result rows. Budget ~4x the section-3 time.

In [ ]:
abl_cmd = [PY, "-m", "src.tutorial_distilbert",
           "--model", MODEL,
           "--seeds", ",".join(str(s) for s in SEEDS),
           "--epochs", str(EPOCHS), "--lr", str(LR),
           "--batch-size", str(BATCH_SIZE), "--weight-decay", str(WEIGHT_DEC),
           "--threshold", str(THRESHOLD), "--max-length", str(MAX_LEN),
           "--task", TASK, "--out", OUT, "--ablation", "--no-demo"]
print(" ".join(abl_cmd), "\n")

proc = subprocess.Popen(abl_cmd, cwd=ROOT, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True,
                        encoding="utf-8", errors="replace", bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)

In [ ]:
abl = pd.read_csv(Path(OUT) / "ablation_summary.csv")
abl = abl[abl["task"] == TASK]
display(abl.round(3))

pivot = abl.pivot(index="loss", columns="threshold_mode", values="macro_f1")
pivot = pivot.reindex([l for l in ["bce", "pos_bce", "focal", "asl",
                                   "ce", "weighted_ce"] if l in pivot.index])
ax = pivot.plot.bar(figsize=(8, 4), rot=0,
                    color={"fixed 0.5": "#C44E52", "tuned": "#4C72B0"})
ax.set_ylabel("val macro-F1")
ax.set_title("Loss x threshold: what each fix is worth on its own")
ax.grid(axis="y", alpha=0.3)
ax.legend(title="threshold")
plt.tight_layout()
plt.show()

base = pivot.loc["bce", "fixed 0.5"]
print(f"tutorial baseline (bce @ 0.5)     : macro-F1 {base:.3f}")
print(f"threshold alone (bce @ tuned)     : {pivot.loc['bce', 'tuned']:.3f}  "
      f"({pivot.loc['bce', 'tuned'] - base:+.3f})")
print(f"loss alone (best loss @ 0.5)      : {pivot['fixed 0.5'].max():.3f}  "
      f"({pivot['fixed 0.5'].max() - base:+.3f})  [{pivot['fixed 0.5'].idxmax()}]")
print(f"both (best loss @ tuned)          : {pivot['tuned'].max():.3f}  "
      f"({pivot['tuned'].max() - base:+.3f})  [{pivot['tuned'].idxmax()}]")
if "mean_labels_predicted" in abl:
    print("\nlabels fired per row (true mean is ~0.79) - this is what separates "
          "under-firing\nfrom spraying, which macro-F1 alone cannot:")
    print(abl.pivot(index="loss", columns="threshold_mode",
                    values="mean_labels_predicted").round(2).to_string())

pr = [c for c in ("macro_precision", "macro_recall", "macro_f1") if c in abl]
if len(pr) == 3:
    print("\nprecision vs recall - F1 is their harmonic mean, so it hides which "
          "way a\nconfiguration is failing:")
    print(abl.set_index(["loss", "threshold_mode"])[pr].round(3).to_string())

## 7. What this tells you

Two things worth writing down after the run:

1. **Where the recipe breaks.** Unweighted BCE plus a flat 0.5 threshold, on a
   corpus where the commonest distortion covers 11.5% of training rows and the
   rarest 5.0%, pushes every sigmoid below 0.5 for the rare classes. Low
   `macro_f1` next to a much healthier `roc_auc_micro` is the signature: the
   model *ranks* correctly but never crosses the threshold. Section 6b
   quantifies which of the two fixes is worth more.

2. **Which fix matters.** If `bce @ tuned` alone recovers most of the gap, the
   model was fine and only the decision line was wrong. If it takes `pos_bce`
   or `asl` to move `macro_f1`, the unweighted loss really did stop it learning
   the rare classes. Those are different findings and worth stating separately
   in a write-up.

3. **The comparison.** The cell below puts this run next to the project's own
   multi-label runs, if you have any in `results/`. Note `roberta-base` is a
   bigger backbone than `distilbert-base`, so that row mixes architecture with
   recipe - section 6b is the controlled comparison, since it holds the
   architecture fixed and varies only the loss.

In [ ]:
mine = pd.DataFrame([{
    "model": f"{MODEL} (tutorial recipe)", "split": "val", "seeds": len(per_seed),
    "micro_f1": summary.loc["micro_f1", "mean"],
    "macro_f1": summary.loc["macro_f1", "mean"],
    "loss": "BCE, unweighted", "threshold": f"fixed {THRESHOLD}",
}])

others = pd.DataFrame()
pcsv = Path("results/paper_comparison.csv")
if pcsv.exists():
    ref = pd.read_csv(pcsv)
    ref = ref[(ref.task == "multilabel") & (ref.split == "val")]
    if not ref.empty:
        others = (ref.groupby("model")
                     .agg(seeds=("seed", "nunique"),
                          micro_f1=("micro_f1", "mean"),
                          macro_f1=("macro_f1", "mean"))
                     .reset_index())
        others["model"] += " (project pipeline)"
        others["split"] = "val"
        others["loss"] = "BCE + pos_weight"
        others["threshold"] = "swept on val"

display(pd.concat([mine, others], ignore_index=True)
          [["model", "split", "seeds", "micro_f1", "macro_f1", "loss", "threshold"]]
          .round(3))

## 8. Collect the results

Builds `docs/RERUN_EXPERIMENTS.md` from whatever has been run, then copies
everything into `/kaggle/working/` so it appears in the session's **Output**
pane and can be downloaded.

Checkpoints are deliberately NOT copied — they are ~250 MB each and the repo
gitignores them. Only re-run the training if you need the weights again.

In [ ]:
# Roll every run in OUT into one document, grouped by task.
r = subprocess.run([PY, "-m", "src.make_rerun_table", "--results", OUT,
                    "--out", "docs/RERUN_EXPERIMENTS.md"],
                   cwd=ROOT, capture_output=True, text=True,
                   encoding="utf-8", errors="replace")
print(r.stdout or r.stderr)

import shutil
KAGGLE_OUT = Path("/kaggle/working")
if KAGGLE_OUT.exists():
    dest = KAGGLE_OUT / OUT
    shutil.copytree(Path(OUT), dest, dirs_exist_ok=True)
    doc = Path("docs/RERUN_EXPERIMENTS.md")
    if doc.exists():
        shutil.copy(doc, KAGGLE_OUT / doc.name)
    print(f"\ncopied to {KAGGLE_OUT} - check the Output pane:")
    for f in sorted(dest.rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(KAGGLE_OUT)}  ({f.stat().st_size:,} b)")
else:
    print(f"\nNot on Kaggle - results are already in {ROOT / OUT}")

# The generated doc, inline.
doc = Path("docs/RERUN_EXPERIMENTS.md")
if doc.exists():
    try:
        from IPython.display import Markdown
        display(Markdown(doc.read_text(encoding="utf-8")))
    except ImportError:
        print(doc.read_text(encoding="utf-8")[:4000])

Artifacts written by this notebook:

```
results_tutorial_distilbert/
  val_summary.csv            mean +/- std over seeds
  val_metrics_per_seed.csv   one row per seed
  val_metrics_<model>_<seed>.json
  per_class_val_mean.csv     precision/recall/F1/support per distortion
  per_class_val_f1.png
  epoch_history_<model>_<seed>.csv
  demo_predictions.csv
checkpoints/tutorial_<model>_<seed>/   weights + meta.json (gitignored)
```